In [1]:
import os
import dotenv
import whisper
# from moviepy.editor import VideoFileClip
from openai import OpenAI
import autogen

In [2]:
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
config_list = [
    {
        "model": "gpt-4",
        "api_key": os.getenv("OPENAI_API_KEY"),
    }
]

In [3]:
def recognize_transcript_from_video(audio_filepath):
    """Transcribe the speech in a video or audio file with Whisper.

    Writes one line per Whisper segment to ``transcription.txt`` in the
    current directory, formatted as ``"<start>s to <end>s: <sentence>"``.

    Args:
        audio_filepath: Path to the video or audio file. Any format that
            ffmpeg can decode (e.g. .mp4, .webm, .mkv) works.

    Returns:
        The contents of ``transcription.txt`` as a string, or an error
        message string if the file is missing or transcription fails.
    """
    try:
        # Load model
        model = whisper.load_model("small")

        # Transcribe audio with detailed timestamps
        result = model.transcribe(audio_filepath, verbose=True)

        # Initialize variables for transcript
        transcript = []
        sentence = ""
        start_time = 0

        # Iterate through the segments in the result
        for segment in result["segments"]:
            # If new sentence starts, save the previous one and reset variables
            if segment["start"] != start_time and sentence:
                transcript.append(
                    {
                        "sentence": sentence.strip() + ".",
                        "timestamp_start": start_time,
                        "timestamp_end": segment["start"],
                    }
                )
                sentence = ""
                start_time = segment["start"]

            # Add the word to the current sentence
            sentence += segment["text"] + " "

        # Add the final sentence
        if sentence:
            transcript.append(
                {
                    "sentence": sentence.strip() + ".",
                    "timestamp_start": start_time,
                    "timestamp_end": result["segments"][-1]["end"],
                }
            )

        # Save the transcript to a file
        with open("transcription.txt", "w") as file:
            for item in transcript:
                sentence = item["sentence"]
                start_time, end_time = item["timestamp_start"], item["timestamp_end"]
                file.write(f"{start_time}s to {end_time}s: {sentence}\n")

        # ag2 only accepts str/number function results, so return the file contents
        with open("transcription.txt", "r") as file:
            return file.read()

    except FileNotFoundError:
        return "The specified audio file could not be found."
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

def translate_text(input_text, source_language, target_language):
    """Translate a piece of text into subtitle-style text using the OpenAI API.

    Args:
        input_text: The text to translate.
        source_language: Language of ``input_text``, e.g. "English".
        target_language: Language to translate into, e.g. "Spanish".

    Returns:
        The translated text, or ``None`` if the API returned no choices.
    """
    client = OpenAI(api_key=key)

    response = client.chat.completions.create(
#        model="gpt-3.5-turbo",
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": f"Directly translate the following {source_language} text to a pure {target_language} "
                f"video subtitle text without additional explanation.: '{input_text}'",
            },
        ],
        max_tokens=1500,
    )

    # Correctly accessing the response content
    translated_text = response.choices[0].message.content if response.choices else None
    return translated_text


def translate_transcript(source_language, target_language):
    """Translate every line of ``transcription.txt``, keeping its timestamps.

    Only the text after each ``"<start>s to <end>s: "`` prefix is sent to
    :func:`translate_text`; lines without that prefix are copied unchanged.
    The result is also saved to ``transcription_<target_language>.txt``.

    Args:
        source_language: Language of the transcript, e.g. "English".
        target_language: Language to translate into, e.g. "Spanish".

    Returns:
        The translated transcript as a single newline-separated string.
    """
    with open("transcription.txt", "r") as f:
        lines = f.readlines()

    translated_transcript = []

    for line in lines:
        # Split each line into timestamp and text parts
        parts = line.strip().split(": ", 1)
        if len(parts) == 2:
            timestamp, text = parts[0], parts[1]
            # Translate only the text part
            translated_text = translate_text(text, source_language, target_language)
            # Reconstruct the line with the translated text and the preserved timestamp
            translated_line = f"{timestamp}: {translated_text}"
            translated_transcript.append(translated_line)
        else:
            # If the line doesn't contain a timestamp, add it as is
            translated_transcript.append(line.strip())

    translated_text = "\n".join(translated_transcript)
    with open(f"transcription_{target_language.lower()}.txt", "w") as f:
        f.write(translated_text)
    return translated_text

In [4]:
llm_config = {
    "functions": [
        {
            "name": "recognize_transcript_from_video",
            "description": "recognize the speech from video and transfer into a txt file",
            "parameters": {
                "type": "object",
                "properties": {
                    "audio_filepath": {
                        "type": "string",
                        "description": "path of the video file",
                    }
                },
                "required": ["audio_filepath"],
            },
        },
        {
            "name": "translate_transcript",
            "description": "using translate_text function to translate the script",
            "parameters": {
                "type": "object",
                "properties": {
                    "source_language": {
                        "type": "string",
                        "description": "source language",
                    },
                    "target_language": {
                        "type": "string",
                        "description": "target language",
                    },
                },
                "required": ["source_language", "target_language"],
            },
        },
    ],
    "config_list": config_list,
    "timeout": 120,
}
source_language = "English"
target_language = "Spanish"
key = os.getenv("OPENAI_API_KEY")
target_video = "The Impact of Wikipedia Gideon Digby.webm - Wikimedia Common.mkv"

In [5]:
chatbot = autogen.AssistantAgent(
    name="chatbot",
    system_message="For coding tasks, only use the functions you have been provided with. Reply TERMINATE when the task is done.",
    llm_config=llm_config,
)

user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config={
        "work_dir": "coding_2",
        "use_docker": False,
    }
)

user_proxy.register_function(
    function_map={
        "recognize_transcript_from_video": recognize_transcript_from_video,
        "translate_transcript": translate_transcript,
    }
)
user_proxy.initiate_chat(
    chatbot,
    message=f"For the video located in {target_video}, recognize the speech and transfer it into a script file, "
    f"then translate from {source_language} text to a {target_language} video subtitle text. ",
)

user_proxy (to chatbot):

For the video located in The Impact of Wikipedia Gideon Digby.webm - Wikimedia Common.mkv, recognize the speech and transfer it into a script file, then translate from English text to a Spanish video subtitle text. 

--------------------------------------------------------------------------------
chatbot (to user_proxy):

***** Suggested function call: recognize_transcript_from_video *****
Arguments: 
{
"audio_filepath": "The Impact of Wikipedia Gideon Digby.webm - Wikimedia Common.mkv"
}
********************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION recognize_transcript_from_video...
Call ID: None
Input arguments: {'audio_filepath': 'The Impact of Wikipedia Gideon Digby.webm - Wikimedia Common.mkv'}


/opt/anaconda3/lib/python3.13/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:06.320]  Sometimes I think I'm a bit crazy to go out in the rain or climb a fence or cross a swollen creek or
[00:06.840 --> 00:08.840]  Yeah, go for a drive for
[00:09.080 --> 00:12.680]  What word that to be almost a thousand miles to take a photograph of one side?
[00:13.600 --> 00:17.600]  My name is Gideon and my user name is Nangara in Australia
[00:17.600 --> 00:21.840]  There's that old fable about the outback and there was a sign for one of the shires
[00:21.840 --> 00:25.800]  I knew I'd seen it before and all it says is outback starts here
[00:26.200 --> 00:30.360]  So I drove from Perth up to the sign took the photo and drove back home again
[00:30.600 --> 00:34.240]  Yeah, it all began with the kangaroo poor right click on the Wikipedia article
[00:34.240 --> 00:38.160]  And it says this is the floral emblem of Western Australia or some

ChatResult(chat_id=104355101813878120961479894418983403723, chat_history=[{'content': 'For the video located in The Impact of Wikipedia Gideon Digby.webm - Wikimedia Common.mkv, recognize the speech and transfer it into a script file, then translate from English text to a Spanish video subtitle text. ', 'role': 'assistant', 'name': 'user_proxy'}, {'function_call': {'arguments': '{\n"audio_filepath": "The Impact of Wikipedia Gideon Digby.webm - Wikimedia Common.mkv"\n}', 'name': 'recognize_transcript_from_video'}, 'content': None, 'role': 'assistant'}, {'content': "0s to 6.84s: Sometimes I think I'm a bit crazy to go out in the rain or climb a fence or cross a swollen creek or.\n6.84s to 9.08s: Yeah, go for a drive for.\n9.08s to 13.6s: What word that to be almost a thousand miles to take a photograph of one side?.\n13.6s to 17.6s: My name is Gideon and my user name is Nangara in Australia.\n17.6s to 21.84s: There's that old fable about the outback and there was a sign for one of the sh